In [1]:
# ============================================================
# Sprint 1 Integration Test
# ============================================================
#
# Tests:
#   1. Load real processed BharatFlux data
#   2. Extract real MOD16A2GF data through the common API
#   3. Temporally align the two datasets
#   4. Merge them for benchmarking
#   5. Calculate benchmark statistics
#   6. Export extraction.csv
#   7. Export benchmark.json
#
# This notebook is intentionally a test notebook.
# It will later be archived under notebooks/experiments/.
# ============================================================


# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# Project path
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

# If notebook is executed from notebooks/experiments/,
# move back to project root.
if PROJECT_ROOT.name == "experiments":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


# ------------------------------------------------------------
# OpenETBench imports
# ------------------------------------------------------------

from extraction.gee import initialize
from extraction.sites import get_site
from extraction.products import get_product
from extraction.extractor import extract_timeseries

from harmonization.merge import merge_observed_satellite
from harmonization.temporal import align_to_common_dates

from benchmarking.metrics import calculate_metrics

from utils.results import (
    ensure_product_result_dir,
    save_extraction,
    save_benchmark,
)


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

SITE_ID = "BFT"
PRODUCT_ID = "MOD16A2GF"

YEAR = 2016

START_DATE = f"{YEAR}-01-01"

# Earth Engine filterDate uses an exclusive end date.
END_DATE = f"{YEAR + 1}-01-01"


# ------------------------------------------------------------
# Initialize Earth Engine
# ------------------------------------------------------------

initialize()


# ------------------------------------------------------------
# 1. Load processed BharatFlux observations
# ------------------------------------------------------------

processed_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bharatflux"
)

observed_path = (
    processed_dir
    / f"{SITE_ID}_{YEAR}_LE_ET_dmean.parquet"
)


print("Observed dataset:")
print(observed_path)


if not observed_path.exists():
    raise FileNotFoundError(
        f"Processed BharatFlux file not found:\n{observed_path}"
    )


observed = pd.read_parquet(
    observed_path
)


print("\nObserved columns:")
print(observed.columns.tolist())

print("\nObserved shape:")
print(observed.shape)

print("\nObserved head:")
display(observed.head())


# ------------------------------------------------------------
# 2. Standardize observation dataframe
# ------------------------------------------------------------

# The processed BharatFlux datasets should already contain
# the canonical ET representation.
#
# We only enforce the columns required by the benchmark API.

required_observed_columns = {
    "DoY",
    "ET",
}

missing = (
    required_observed_columns
    - set(observed.columns)
)

if missing:
    raise ValueError(
        f"Observed dataset is missing columns: {missing}"
    )


observed = observed.copy()

observed["DoY"] = (
    pd.to_numeric(
        observed["DoY"],
        errors="coerce",
    )
    .astype("Int64")
)

observed["ET"] = pd.to_numeric(
    observed["ET"],
    errors="coerce",
)

observed = observed.dropna(
    subset=[
        "DoY",
        "ET",
    ]
)


# ------------------------------------------------------------
# 3. Get site and product from registry
# ------------------------------------------------------------

site = get_site(
    SITE_ID
)

product = get_product(
    PRODUCT_ID
)


print("\nSite:")
print(site)

print("\nProduct:")
print(product)


# ------------------------------------------------------------
# 4. Extract MOD16A2GF through common API
# ------------------------------------------------------------

print("\nExtracting satellite ET...")

satellite = extract_timeseries(
    site,
    product,
    START_DATE,
    END_DATE,
)


print("\nSatellite shape:")
print(satellite.shape)

print("\nSatellite head:")
display(satellite.head())


# ------------------------------------------------------------
# 5. Temporal harmonization
# ------------------------------------------------------------

# MOD16A2GF is an 8-day product.
# BharatFlux is daily.
#
# The current harmonization layer aligns both datasets
# using common DoY values.

observed_aligned, satellite_aligned = align_to_common_dates(observed, satellite)

print("\nAligned observations:")
print(observed_aligned.shape)

print("\nAligned satellite:")
print(satellite_aligned.shape)


# ------------------------------------------------------------
# 6. Create benchmark-ready dataframe
# ------------------------------------------------------------

merged = merge_observed_satellite(
    observed_aligned,
    satellite_aligned,
)


print("\nBenchmark dataframe:")
print(merged.shape)

display(
    merged.head()
)


# ------------------------------------------------------------
# 7. Calculate benchmark statistics
# ------------------------------------------------------------

report = calculate_metrics(
    merged
)


print("\nBenchmark statistics")
print("====================")

print(
    f"RMSE        : {report.rmse:.4f}"
)

print(
    f"MAE         : {report.mae:.4f}"
)

print(
    f"Bias        : {report.bias:.4f}"
)

print(
    f"Correlation : {report.correlation:.4f}"
)

print(
    f"R²          : {report.r2:.4f}"
)


# ------------------------------------------------------------
# 8. Create canonical result directory
# ------------------------------------------------------------

paths = ensure_product_result_dir(
    site_id=SITE_ID,
    product_id=PRODUCT_ID,
)


print("\nResult directory:")
print(paths.root)


# ------------------------------------------------------------
# 9. Save extraction.csv
# ------------------------------------------------------------

extraction_path = save_extraction(
    merged,
    paths,
)


print("\n✓ extraction.csv saved:")
print(extraction_path)


# ------------------------------------------------------------
# 10. Save benchmark.json
# ------------------------------------------------------------

benchmark_path = save_benchmark(
    report,
    paths,
    site=SITE_ID,
    product=PRODUCT_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    n=len(merged),
)


print("\n✓ benchmark.json saved:")
print(benchmark_path)


# ------------------------------------------------------------
# 11. Final validation
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("SPRINT 1 VALIDATION")
print("=" * 60)

assert paths.root.exists()

assert paths.extraction.exists()

assert paths.benchmark.exists()

assert len(merged) > 0

assert {
    "Date",
    "DoY",
    "Observed_ET",
    "Satellite_ET",
}.issubset(
    merged.columns
)

print("✓ Real BharatFlux data loaded")

print("✓ MOD16A2GF extracted through common API")

print("✓ Temporal alignment completed")

print("✓ Benchmark dataframe created")

print("✓ Benchmark statistics calculated")

print("✓ extraction.csv exported")

print("✓ benchmark.json exported")

print("\nSprint 1 PASSED.")

✓ Earth Engine initialized successfully.
Observed dataset:
E:\GRAVITY\IITM-Pune\OpenETBench\data\processed\bharatflux\BFT_2016_LE_ET_dmean.parquet

Observed columns:
['DoY', 'LE', 'ET']

Observed shape:
(366, 3)

Observed head:


,DoY,LE,ET
0,1,121.533229,4.666875
1,2,124.119004,4.766169
2,3,104.577071,4.015759
3,4,109.743590,4.214153
4,5,110.055843,4.226144



Site:
Site(id='BFT', latitude=21.86, longitude=77.42, elevation=507, buffer_m=700)

Product:
ETProduct(name='MOD16A2GF', collection='MODIS/061/MOD16A2GF', band='ET', scale_factor=0.1, spatial_resolution=500, temporal_resolution='8-day', units='mm/8-day', provider='NASA', coverage='Global', product_type='Remote Sensing', aggregation='native', sampling='mean')

Extracting satellite ET...

Satellite shape:
(46, 3)

Satellite head:


,Date,DoY,ET
0,2016-01-01,1,3.793601
1,2016-01-09,9,3.947705
2,2016-01-17,17,4.724370
3,2016-01-25,25,2.116354
4,2016-02-02,33,2.866710



Aligned observations:
(46, 3)

Aligned satellite:
(46, 3)

Benchmark dataframe:
(46, 5)


,Date,DoY,Observed_LE,Observed_ET,Satellite_ET
0,2016-01-01,1,121.533229,4.666875,3.793601
1,2016-01-09,9,107.454253,4.126243,3.947705
2,2016-01-17,17,134.438082,5.162422,4.724370
3,2016-01-25,25,87.810117,3.371908,2.116354
4,2016-02-02,33,64.205003,2.465472,2.866710



Benchmark statistics
RMSE        : 13.1718
MAE         : 8.9248
Bias        : 7.6164
Correlation : 0.7880
R²          : 0.6210

Result directory:
E:\GRAVITY\IITM-Pune\OpenETBench\results\BFT\MOD16A2GF

✓ extraction.csv saved:
E:\GRAVITY\IITM-Pune\OpenETBench\results\BFT\MOD16A2GF\extraction.csv

✓ benchmark.json saved:
E:\GRAVITY\IITM-Pune\OpenETBench\results\BFT\MOD16A2GF\benchmark.json


SPRINT 1 VALIDATION
✓ Real BharatFlux data loaded
✓ MOD16A2GF extracted through common API
✓ Temporal alignment completed
✓ Benchmark dataframe created
✓ Benchmark statistics calculated
✓ extraction.csv exported
✓ benchmark.json exported

Sprint 1 PASSED.
